# ScienceQA Visual Challenge: Starter Notebook

This notebook is configured for Google Colab and now delegates experiment logic to the repo scripts.
Run the setup cell first, then use the launcher cells below for single runs, stage sweeps, and result summaries.


In [ ]:
# ── 0. Colab setup ───────────────────────────────────────────────
from pathlib import Path
import subprocess
import ipywidgets as widgets
from IPython.display import display

REPO_URL = "https://github.com/Demetri65/dl-kaggle-competition-final.git"
REPO_REF = "main"
REPO_DIR = Path("/content/dl-kaggle-competition-final")
SOURCE_ENV = Path("/content/.env")
UPLOADER_KEY = "_env_uploader"

def get_uploaded_file(uploader):
    value = uploader.value

    if isinstance(value, dict):
        filename, uploaded_file = next(iter(value.items()))
        if isinstance(uploaded_file, dict):
            content = uploaded_file.get("content", uploaded_file.get("data"))
        else:
            content = uploaded_file
    else:
        uploaded_file = value[0]
        if isinstance(uploaded_file, dict):
            filename = uploaded_file["name"]
            content = uploaded_file["content"]
        else:
            filename = uploaded_file.name
            content = uploaded_file.content

    payload = content.tobytes() if hasattr(content, "tobytes") else bytes(content)
    return filename, payload

ready_to_bootstrap = SOURCE_ENV.exists()

if ready_to_bootstrap:
    print(f"Using existing {SOURCE_ENV}")
else:
    uploader = globals().get(UPLOADER_KEY)
    if uploader is None:
        uploader = widgets.FileUpload(accept=".env", multiple=False, description="Upload .env")
        globals()[UPLOADER_KEY] = uploader

    if not uploader.value:
        display(uploader)
        print("Select your local .env file in the upload widget above, then rerun this cell.")
    else:
        filename, payload = get_uploaded_file(uploader)
        SOURCE_ENV.write_bytes(payload)
        SOURCE_ENV.chmod(0o600)
        print(f"Saved {filename} to {SOURCE_ENV}")
        uploader.close()
        globals().pop(UPLOADER_KEY, None)
        ready_to_bootstrap = True

if ready_to_bootstrap:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)

    print(f"Repo synced to latest origin/{REPO_REF} at {REPO_DIR}")
    result = subprocess.run(
        ["bash", "scripts/bootstrap_colab.sh"],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, end="")
        raise RuntimeError(f"scripts/bootstrap_colab.sh failed with exit code {result.returncode}")


## 1. Parameters

Set the experiment ids and overrides here. Use `EVAL_ARTIFACT_DIR` for true eval-only runs backed by a saved `outputs/<run_name>/` directory.


In [ ]:
# Single-run parameters
EXPERIMENT_ID = "f02_multimodal_index"
EXPERIMENT_OVERRIDES = []
SEED = None
OUTPUT_DIR = None
PREDICT_TEST = False
FINAL_RETRAIN = False
EVAL_ARTIFACT_DIR = None

# Stage-run parameters
STAGE_EXPERIMENTS = [
    "f02_multimodal_index",
    "f03_multimodal_letter",
]
STAGE_OVERRIDES = []
STAGE_NAME = None
STAGE_SELECT_BEST_BY = "val_accuracy"
STAGE_ASCENDING = False
STAGE_ENSEMBLE_NAME = None

# Results summary parameters
SUMMARY_SORT_BY = "val_accuracy"
SUMMARY_TOP = 10


## 2. Helpers


In [ ]:
import shlex
import subprocess
from pathlib import Path

REPO_ROOT = Path("/content/dl-kaggle-competition-final")

def extend_with_overrides(args, overrides):
    for override in overrides:
        args.extend(["--set", override])

def run_repo_command(args):
    command = ["python3", *args]
    print("$", " ".join(shlex.quote(part) for part in command))
    result = subprocess.run(
        command,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


## 3. Run One Experiment


In [ ]:
run_overrides = list(EXPERIMENT_OVERRIDES)
if EVAL_ARTIFACT_DIR:
    run_overrides.extend([
        "training.epochs=0",
        f"runtime.eval_artifact_dir={EVAL_ARTIFACT_DIR}",
    ])

run_args = ["scripts/run_experiment.py", "--experiment", EXPERIMENT_ID]
extend_with_overrides(run_args, run_overrides)
if SEED is not None:
    run_args.extend(["--seed", str(SEED)])
if OUTPUT_DIR:
    run_args.extend(["--output-dir", OUTPUT_DIR])
if PREDICT_TEST:
    run_args.append("--predict-test")
if FINAL_RETRAIN:
    run_args.append("--final-retrain")

run_repo_command(run_args)


## 4. Run A Stage


In [ ]:
stage_args = ["scripts/run_stage.py", *STAGE_EXPERIMENTS]
extend_with_overrides(stage_args, STAGE_OVERRIDES)
if SEED is not None:
    stage_args.extend(["--seed", str(SEED)])
if OUTPUT_DIR:
    stage_args.extend(["--output-dir", OUTPUT_DIR])
if PREDICT_TEST:
    stage_args.append("--predict-test")
if FINAL_RETRAIN:
    stage_args.append("--final-retrain")
if STAGE_NAME:
    stage_args.extend(["--stage-name", STAGE_NAME])
if STAGE_ENSEMBLE_NAME:
    stage_args.extend(["--ensemble-name", STAGE_ENSEMBLE_NAME])
if STAGE_SELECT_BEST_BY:
    stage_args.extend(["--select-best-by", STAGE_SELECT_BEST_BY])
if STAGE_ASCENDING:
    stage_args.append("--ascending")

run_repo_command(stage_args)


## 5. Summarize Results


In [ ]:
summary_args = [
    "scripts/summarize_results.py",
    "--sort-by", SUMMARY_SORT_BY,
    "--top", str(SUMMARY_TOP),
]

run_repo_command(summary_args)
